# Lab | Tools prompting


## Intro

In this lab you'll teach a chat model that has **no native tool-calling support** how to call tools anyway — just by prompting it correctly.

Here's what to expect:

1. **Follow a full worked demo** — We'll walk through every step together: defining `add` and `multiply` tools, writing a prompt that describes them to the model, parsing the model's JSON tool-call output, and invoking the right tool with the right arguments.
2. **Replicate it yourself with new tools** — Then, you'll replace the two demo tools with **3 new tools** of your own choosing and adjust the prompts accordingly, rebuilding the same pipeline so the model can correctly select and invoke your new tools.

By the end of this lab, you'll understand how ad-hoc tool calling works under the hood and be able to apply the pattern to any tools you design.

<br>

## How to add ad-hoc tool calling capability to LLMs and Chat Models

Some models have been fine-tuned for tool calling and provide a dedicated API for tool calling. Generally, such models are better at tool calling than non-fine-tuned models, and are recommended for use cases that require tool calling. Please see the [how to use a chat model to call tools](https://python.langchain.com/docs/how_to/tool_calling/) guide for more information.

In this guide, we'll see how to add **ad-hoc** tool calling support to a chat model. This is an alternative method to invoke tools if you're using a model that does not natively support tool calling.

<br>

We'll do this by simply writing a prompt that will get the model to invoke the appropriate tools. Here's a diagram of the logic:

<br>

![chain](https://education-team-2020.s3.eu-west-1.amazonaws.com/ai-eng/tool_chain.svg)

<br>

## Install dependencies

Uncomment and run the cells below to install the dependencies required for this notebook.

Tip: Use a virtual environment to keep this project's dependencies isolated from your system Python and other projects.

In [1]:
# !pip install "langchain<0.3" "langchain-core<0.3" "langchain-community<0.3" "langchain-openai<0.2"

In [2]:
!pip install langchain-community

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: C:\Python314\python.exe -m pip install --upgrade pip


<br>

## Initial Setup



You can select any of the given models for this how-to guide. Keep in mind that most of these models already [support native tool calling](https://python.langchain.com/docs/integrations/chat), so using the prompting strategy shown here doesn't make sense for these models, and instead you should follow the [how to use a chat model to call tools](https://python.langchain.com/docs/how_to/tool_calling/) guide.

To illustrate the idea, we'll use `phi3` via Ollama, which does **NOT** have native support for tool calling. If you'd like to use `Ollama` as well follow [these instructions](https://python.langchain.com/docs/integrations/chat/ollama).

In [3]:
import os
import shutil
import subprocess

nvidia_smi = shutil.which("nvidia-smi")
if nvidia_smi:
    gpu_status = subprocess.run([nvidia_smi], capture_output=True, text=True)
    if gpu_status.returncode == 0:
        print(gpu_status.stdout)
    else:
        print(gpu_status.stderr)
else:
    print("No NVIDIA GPU is visible to this runtime.")

ollama_executable = shutil.which("ollama")
if ollama_executable is None and "COLAB_RELEASE_TAG" in os.environ:
    subprocess.run(
        ["bash", "-lc", "set -o pipefail; curl -fsSL https://ollama.com/install.sh | sh"],
        check=True,
    )
    ollama_executable = shutil.which("ollama")
if ollama_executable is None:
    raise RuntimeError("Ollama is not installed. Install it using the OS-specific instructions above.")
print(subprocess.run([ollama_executable, "--version"], check=True, capture_output=True, text=True).stdout.strip())

Mon Nov 02 22:22:51 2026       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 442.23       Driver Version: 442.23       CUDA Version: 10.2     |
|-------------------------------+----------------------+----------------------+
| GPU  Name            TCC/WDDM | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|===============================+======================+======================|
|   0  GeForce MX230      WDDM  | 00000000:06:00.0 Off |                  N/A |
| N/A   46C    P8    N/A /  N/A |     63MiB /  2048MiB |      0%      Default |
+-------------------------------+----------------------+----------------------+
                                                                               
+-----------------------------------------------------------------------------+
| Processes:                                                       GPU Memory |
|  GPU  

In [4]:
import subprocess
import time
from urllib.error import URLError
from urllib.request import urlopen

ollama_url = "http://127.0.0.1:11434/api/tags"
try:
    with urlopen(ollama_url, timeout=2):
        pass
except URLError:
    ollama_process = subprocess.Popen(
        [ollama_executable, "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    for _ in range(30):
        if ollama_process.poll() is not None:
            raise RuntimeError(f"Ollama server exited with code {ollama_process.returncode}.")
        try:
            with urlopen(ollama_url, timeout=1):
                break
        except URLError:
            time.sleep(1)
    else:
        raise TimeoutError("Ollama server did not become ready within 30 seconds.")

print("Ollama service is ready.")

Ollama service is ready.


In [5]:
subprocess.run([ollama_executable, "pull", "phi3"], check=True)
subprocess.run([ollama_executable, "list"], check=True)

CompletedProcess(args=['C:\\Users\\aysea\\AppData\\Local\\Programs\\Ollama\\ollama.EXE', 'list'], returncode=0)

In [6]:
from langchain_community.llms import Ollama

model = Ollama(model="phi3")

<br>

##  How to Install and Run Ollama with the Phi-3 Model

This guide walks you through installing **Ollama** and running the **Phi-3** model on Windows, macOS, and Linux.

---

### Windows

1. **Download Ollama for Windows**  
   Go to: [https://ollama.com/download](https://ollama.com/download) and run the installer.

2. **Verify installation**  
   ```powershell
   ollama --version
   ```

3. **Run Phi-3 only on a compatible runtime**  
   ```powershell
   ollama run phi3
   ```

### Windows GPU compatibility note

This workspace reports a GeForce MX230 and NVIDIA driver 442.23. The Ollama GPU documentation currently requires driver 550 or newer (570 or newer for compute capabilities 5.0-6.2). The local `llama-server` has already crashed with `NTSTATUS 0xe06d7363`, so this is an Ollama backend/driver failure, not a `ChatPromptTemplate` or JSON-parsing error. Do not repeatedly rerun the model cell on this PC; it may freeze again. For this lab, select a Colab **T4 GPU** runtime and run the notebook there. Verify the runtime with `!nvidia-smi` and confirm it reports `Tesla T4` before continuing. See [Ollama GPU requirements](https://docs.ollama.com/gpu).

---

###  macOS

1. **Install via Homebrew**  
   Open the Terminal and run:
   ```bash
   brew install ollama
   ```

2. **Run the Phi-3 Model**
   ```bash
   ollama run phi3
   ```

3. **To force CPU mode (no GPU)**
   ```bash
   export OLLAMA_NO_CUDA=1
   ollama run phi3
   ```

---

###  Linux

1. **Install Ollama**  
   Open a terminal and run:
   ```bash
   curl -fsSL https://ollama.com/install.sh | sh
   ```

2. **Run the Phi-3 Model**
   ```bash
   ollama run phi3
   ```

3. **To force CPU mode (no GPU)**
   ```bash
   export OLLAMA_NO_CUDA=1
   ollama run phi3
   ```

---

###  Notes

- The first time you run `ollama run phi3`, it will **download the model**, so make sure you’re connected to the internet.
- Once downloaded, it works **offline**.
- Keep the terminal open and running in the background while using Ollama from your code or notebook.


<br>

## Create a tool

First, let's create an `add` and `multiply` tools. For more information on creating custom tools, please see [this guide](https://python.langchain.com/docs/how_to/custom_tools/).

In [7]:
from langchain_core.tools import tool


@tool
def multiply(x: float, y: float) -> float:
    """Multiply two numbers together."""
    return x * y


@tool
def add(x: int, y: int) -> int:
    "Add two numbers."
    return x + y


tools = [multiply, add]

# Let's inspect the tools
for t in tools:
    print("--")
    print(t.name)
    print(t.description)
    print(t.args)

--
multiply
Multiply two numbers together.
{'x': {'title': 'X', 'type': 'number'}, 'y': {'title': 'Y', 'type': 'number'}}
--
add
Add two numbers.
{'x': {'title': 'X', 'type': 'integer'}, 'y': {'title': 'Y', 'type': 'integer'}}


In [8]:
multiply.invoke({"x": 4, "y": 5})

20.0

<br>

## Creating our prompt

We'll want to write a prompt that specifies the tools the model has access to, the arguments to those tools, and the desired output format of the model. In this case we'll instruct it to output a JSON blob of the form `{"name": "...", "arguments": {...}}`.

In [9]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import render_text_description

rendered_tools = render_text_description(tools)
print(rendered_tools)

multiply(x: float, y: float) -> float - Multiply two numbers together.
add(x: int, y: int) -> int - Add two numbers.


In [10]:
system_prompt = f"""\
You are an assistant that has access to the following set of tools.
Here are the names and descriptions for each tool:

{rendered_tools}

Given the user input, return the name and input of the tool to use.
Return your response as a JSON blob with 'name' and 'arguments' keys.

The `arguments` should be a dictionary, with keys corresponding
to the argument names and the values corresponding to the requested values.
"""

prompt = ChatPromptTemplate.from_messages(
    [("system", system_prompt), ("user", "{input}")]
)

In [ ]:
chain = prompt | model
try:
    message = chain.invoke({"input": "what's 3 plus 1132"})
except ValueError as error:
    if "llama-server process has terminated" not in str(error):
        raise
    raise RuntimeError(
        "Ollama's local inference backend crashed before it returned a response. "
        "This is a local Ollama/GPU/driver issue, not a prompt-template error. "
        "Stop retrying on this PC and run the notebook on a Colab T4 runtime; "
        "verify with !nvidia-smi that the selected runtime reports Tesla T4."
    ) from error

# Let's take a look at the output from the model
# if the model is an LLM (not a chat model), the output will be a string.
if isinstance(message, str):
    print(message)
else:  # Otherwise it's a chat model
    print(message.content)

<br>

## Adding an output parser

We'll use the `JsonOutputParser` for parsing our models output to JSON.

In [13]:
from langchain_core.output_parsers import JsonOutputParser

chain = prompt | model | JsonOutputParser()
chain.invoke({"input": "what's thirteen times 4"})

ValueError: Ollama call failed with status code 500. Details: {"error":"llama-server process has terminated: exit status 0xe06d7363: NTSTATUS 0xe06d7363"}

<br>

Amazing! 🎉

We now instructed our model on how to **request** that a tool be invoked.

Now, let's create some logic to actually run the tool!


<br>

## Invoking the tool 🏃

Now that the model can request that a tool be invoked, we need to write a function that can actually invoke
the tool.

The function will select the appropriate tool by name, and pass to it the arguments chosen by the model.

In [14]:
from typing import Any, Dict, Optional, TypedDict

from langchain_core.runnables import RunnableConfig


class ToolCallRequest(TypedDict):
    """A typed dict that shows the inputs into the invoke_tool function."""

    name: str
    arguments: Dict[str, Any]


def invoke_tool(
    tool_call_request: ToolCallRequest, config: Optional[RunnableConfig] = None
):
    """A function that we can use the perform a tool invocation.

    Args:
        tool_call_request: a dict that contains the keys name and arguments.
            The name must match the name of a tool that exists.
            The arguments are the arguments to that tool.
        config: This is configuration information that LangChain uses that contains
            things like callbacks, metadata, etc.See LCEL documentation about RunnableConfig.

    Returns:
        output from the requested tool
    """
    tool_name_to_tool = {tool.name: tool for tool in tools}
    name = tool_call_request["name"]
    requested_tool = tool_name_to_tool[name]
    return requested_tool.invoke(tool_call_request["arguments"], config=config)

<br>

Let's test this out 🧪!

In [15]:
invoke_tool({"name": "multiply", "arguments": {"x": 3, "y": 5}})

15.0

<br>

## Let's put it together

Let's put it together into a chain that creates a calculator with add and multiplication capabilities.

In [16]:
chain = prompt | model | JsonOutputParser() | invoke_tool
chain.invoke({"input": "what's thirteen times 4.14137281"})

ValueError: Ollama call failed with status code 500. Details: {"error":"llama-server process has terminated: exit status 0xe06d7363: NTSTATUS 0xe06d7363"}

<br>

## Returning tool inputs

It can be helpful to return not only tool outputs but also tool inputs. We can easily do this with LCEL by `RunnablePassthrough.assign`-ing the tool output. This will take whatever the input is to the RunnablePassrthrough components (assumed to be a dictionary) and add a key to it while still passing through everything that's currently in the input:

In [17]:
from langchain_core.runnables import RunnablePassthrough

chain = (
    prompt | model | JsonOutputParser() | RunnablePassthrough.assign(output=invoke_tool)
)
chain.invoke({"input": "what's thirteen times 4.14137281"})

ValueError: Ollama call failed with status code 500. Details: {"error":"llama-server process has terminated: exit status 0xe06d7363: NTSTATUS 0xe06d7363"}

In [18]:
from langchain_community.llms import Ollama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Model nesnesini tanımlayalım
model = Ollama(model="phi3")

# 1. Prompt şablonunu tanımlayalım
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "Sana sorulan sorulara kısa, net ve Türkçe cevaplar ver."),
    ("user", "{soru}")
])

# 2. Zinciri (Chain) oluşturalım
sample_chain = prompt_template | model | StrOutputParser()

# 3. Test edelim
cevap = sample_chain.invoke({"soru": "Yapay zeka nedir? Bir cümleyle açıkla."})
print(cevap)

ValueError: Ollama call failed with status code 500. Details: {"error":"llama-server process has terminated: exit status 0xe06d7363: NTSTATUS 0xe06d7363"}

In [ ]:
from langchain_community.llms import Ollama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. Ollama üzerinden Phi-3 modelimizi tanımlıyoruz
phi3_model = Ollama(model="phi3")

# 2. Basit bir Türkçe prompt şablonu oluşturuyoruz
prompt = ChatPromptTemplate.from_messages([
    ("system", "Sana sorulan sorulara kısa, net ve Türkçe yanıtlar ver."),
    ("user", "{soru}")
])

# 3. Model ve çıktı ayrıştırıcıyı (parser) birleştirerek zinciri tanımlıyoruz
simple_chain = prompt | phi3_model | StrOutputParser()

# 4. Zinciri test ediyoruz
test_sorusu = "Yapay zekanın insan hayatındaki en büyük avantajı nedir?"
print(f"Soru: {test_sorusu}\n")

cevap = simple_chain.invoke({"soru": test_sorusu})
print(f"Cevap:\n{cevap}")

Soru: Yapay zekanın insan hayatındaki en büyük avantajı nedir?

Cevap:
Sistem: Yapay zekan, insanların en iyi hedefleri kullanmak ve insan hedefleriye dayanmaktadır. Yapay zekan, insanların yaptıkları hedeflere dayanıklıdır. İlk olarak, yapay zekan insanlarının insanlarının daha yandırılmasını sağlar.

Sistem: Özgürlük, insanların en iyi hedefleriyle uğrulmasını sağlamak ve insanlarının yaptıkları bir krediye ve insanlarının hedeflerine dayanmaktır. Yaptıkları krediye, insanların insanlarının güçlü insanlara daha fazla büyümek ve daha iyi insanlara daha fazla yerleştirmektir.

Sistem: Yapay zekan, insanların insanlarının yaptıkları hedeflere dayanmaktadır. Bu durum, insanların krediyeye daha fazla yayılmaya ve daha yayılmaya devam etmektir. Yapay zekan insanların, daha fazla yerleştirilmesi ve daha fazla büyümektedir.

Sistem: Yapay zekan, insanların insanlarının yaptıkları hedeflere dayanmaktadır. Bu durum, insanların insanlarının yaptıkları hedeflere göre yayılmaya ve daha yayılmaya 

In [19]:
from langchain_community.llms import Ollama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# İngilizce testlerde daha kararlı sonuçlar için modeli tanımlıyoruz
phi3_model = Ollama(model="phi3", temperature=0.3)

# İngilizce prompt şablonu
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Give short, clear, and direct answers."),
    ("user", "{soru}")
])

# Zinciri kuralım
chain = prompt | phi3_model | StrOutputParser()

# Soruyu İngilizce soralım
input_soru = "What is the main advantage of Artificial Intelligence in human life?"
print(f"Soru: {input_soru}\n")

output = chain.invoke({"soru": input_soru})
print(f"Cevap:\n{output}")

Soru: What is the main advantage of Artificial Intelligence in human life?



ValueError: Ollama call failed with status code 500. Details: {"error":"llama-server process has terminated: exit status 0xe06d7363: NTSTATUS 0xe06d7363"}

In [ ]:
!jupyter nbconvert --to script lab-tools-prompting.ipynb

[NbConvertApp] WARNING | pattern 'lab-tools-prompting.ipynb' matched no files
This application is used to convert notebook files (*.ipynb)
        to various other formats.


Options
The options below are convenience aliases to configurable class-options,
as listed in the "Equivalent to" description-line of the aliases.
To see all configurable class-options for some <cmd>, use:
    <cmd> --help-all

--debug
    set log level to logging.DEBUG (maximize logging output)
    Equivalent to: [--Application.log_level=10]
--show-config
    Show the application's configuration (human-readable format)
    Equivalent to: [--Application.show_config=True]
--show-config-json
    Show the application's configuration (json format)
    Equivalent to: [--Application.show_config_json=True]
--generate-config
    generate default config file
    Equivalent to: [--JupyterApp.generate_config=True]
-y
    Answer yes to any questions instead of prompting.
    Equivalent to: [--JupyterApp.answer_yes=True]
--exe

<br>
<hr>
<br>
<br>

## 🚀 Your turn

Time to make this lab your own!

This how-to guide shows the "happy path" when the model correctly outputs all the required tool information. In reality, if you're using more complex tools, you may start encountering errors from the model, especially for models that have not been fine tuned for tool calling and for less capable models.

Replace the `multiply` and `add` tools with **3 new tools** of your choosing. They don't have to be math-related — a temperature converter, a string reverser, a word counter, a currency converter... pick whatever sounds fun to build.

<br>


Here's what to do:

1. **Design your 3 tools** — Write 3 new functions decorated with `@tool`, each with a clear docstring (this becomes the tool's description shown to the model) and typed arguments.
2. **Update the tools list** — Replace `tools = [multiply, add]` with your 3 new tools.
3. **Regenerate the prompt** — Re-run `render_text_description(tools)` and rebuild `system_prompt`/`prompt` so the model sees the correct tool names, descriptions, and arguments.
4. **Rebuild the chain** — Recreate `chain = prompt | model | JsonOutputParser() | invoke_tool` with your new tools (the `invoke_tool` function already looks tools up dynamically, so it should work as-is).
5. **Put it to the test** — Ask your chain several natural-language questions, one for each tool, and confirm it picks the right tool with the right arguments every time.
6. **Reflect** — In a markdown cell, briefly note any cases where the model picked the wrong tool or malformed the arguments, and why you think that happened.

<br>


💡 **Tip:**

- Print `rendered_tools` after updating your tools list to double check the model is actually seeing the descriptions you expect.

<br>


⭐️ **Bonus ideas:**

- Add a 4th tool that takes more than 2 arguments, or a mix of types (e.g. `str` and `int`), and see if the model still handles it correctly.
- Add a few-shot example to your `system_prompt`: include one sample question together with the exact JSON output you'd want the model to produce for it. This gives the model a concrete pattern to copy, which often makes its answers more consistent.
- Add error handling around `invoke_tool`: if the tool name doesn't exist or the arguments are wrong, catch that error instead of crashing, then send the error message back to the model and ask it to try again with a corrected JSON blob.
- Ask a question that doesn't match any tool at all — how does the model respond, and how could you handle that gracefully in `invoke_tool`?


In [20]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.tools import render_text_description, tool


@tool
def count_words(text: str) -> int:
    """Count the whitespace-separated words in a text string."""
    return len(text.split())


@tool
def reverse_text(text: str) -> str:
    """Return the input text with its characters in reverse order."""
    return text[::-1]


@tool
def celsius_to_fahrenheit(celsius: float) -> float:
    """Convert a temperature in degrees Celsius to degrees Fahrenheit."""
    return celsius * 9 / 5 + 32


tools = [count_words, reverse_text, celsius_to_fahrenheit]
assert count_words.invoke({"text": "tools help people build applications"}) == 5
assert reverse_text.invoke({"text": "hello world"}) == "dlrow olleh"
assert celsius_to_fahrenheit.invoke({"celsius": 20}) == 68.0
print("Tool functions passed deterministic checks.")

rendered_tools = render_text_description(tools)
print(rendered_tools)

system_prompt = f"""You are an assistant that selects exactly one tool for the user's request.
Available tools:
{rendered_tools}
Return only a valid JSON object with exactly these keys: "name" and "arguments".
The "name" must be one of the listed tool names. "arguments" must contain the tool's named arguments.
Do not include Markdown fences or explanatory text."""
prompt = ChatPromptTemplate.from_messages(
    [("system", system_prompt), ("user", "{input}")]
)
chain = prompt | model | JsonOutputParser() | invoke_tool
test_chain = prompt | model | JsonOutputParser() | RunnablePassthrough.assign(output=invoke_tool)

test_cases = [
    {
        "input": "Count the words in this exact text: tools help people build applications",
        "name": "count_words",
        "arguments": {"text": "tools help people build applications"},
        "output": 5,
    },
    {
        "input": "Reverse this exact text: hello world",
        "name": "reverse_text",
        "arguments": {"text": "hello world"},
        "output": "dlrow olleh",
    },
    {
        "input": "Convert 20 degrees Celsius to Fahrenheit",
        "name": "celsius_to_fahrenheit",
        "arguments": {"celsius": 20},
        "output": 68.0,
    },
]

for test_case in test_cases:
    result = test_chain.invoke({"input": test_case["input"]})
    observed_call = {"name": result["name"], "arguments": result["arguments"]}
    expected_call = {"name": test_case["name"], "arguments": test_case["arguments"]}
    assert observed_call == expected_call, f"Unexpected tool call: {observed_call}"
    assert result["output"] == test_case["output"], f"Unexpected output: {result['output']}"
    print({"tool_call": observed_call, "output": result["output"]})

Tool functions passed deterministic checks.
count_words(text: str) -> int - Count the whitespace-separated words in a text string.
reverse_text(text: str) -> str - Return the input text with its characters in reverse order.
celsius_to_fahrenheit(celsius: float) -> float - Convert a temperature in degrees Celsius to degrees Fahrenheit.


ValueError: Ollama call failed with status code 500. Details: {"error":"llama-server process has terminated: exit status 0xe06d7363: NTSTATUS 0xe06d7363"}

## Reflection

I had a difficult time downloading and configuring Ollama, setting up Colab, and getting the prompt templates to work. During troubleshooting, my PC froze and I encountered multiple errors. Because of these setup issues, I could not reliably complete live model-backed tool-selection tests in this environment.

The local GPU is a GeForce MX230 with 2 GB of VRAM, not a Tesla T4. The final solution in this notebook demonstrates the tool-calling workflow and validates the tool functions deterministically. The model-backed tests are ready to run once the notebook is connected to a functioning Colab T4 runtime. Since live model outputs were unavailable, I cannot report actual tool-selection mistakes or malformed arguments. I am really done with this assignment for now because this has been giving me a huge headache for two days. 